# Unit 4 / Chapter 4: Quantum Optimization for AI

> **Main Learning Objective:** Learn how quantum methods optimize the kinds of objective functions that show up in AI: loss surfaces, ground states, combinatorial problems, and more.

| Section | Topic | Why it matters for AI |
|---|---|---|
| 4.1 | Optimization in machine learning | Every model is trained by minimizing a loss |
| 4.2 | Variational Quantum Eigensolver | Finding minimum-energy / minimum-loss configurations |
| 4.3 | Quantum Approximate Optimization Algorithm | Combinatorial problems: features, graphs, inference |
| 4.4 | Applications to AI systems | Where these methods plug into real ML pipelines |

---

## Setup

In [ ]:
# Verify libraries. Works in classic Jupyter, JupyterLite, Pyodide, and Colab.
import importlib.util
required = ["numpy", "matplotlib"]
missing = [p for p in required if importlib.util.find_spec(p) is None]
if missing:
    try:
        import piplite
        await piplite.install(missing)
    except ImportError:
        try:
            import micropip
            await micropip.install(missing)
        except ImportError:
            ip = get_ipython()
            ip.run_line_magic('pip', 'install --quiet ' + ' '.join(missing))
import numpy, matplotlib
print("All libraries ready.")

---
## Course check-in

This logs that you started **Unit 4**. You will be asked to enter the email you signed up with so we can track your progress and email your certificate when you finish all units.

In [ ]:
# ============================================================
# COURSE TRACKING - do not edit
# ============================================================
import json
from urllib.request import Request, urlopen
from urllib.error  import URLError

UNIT_NUMBER = 4
TRACKER_URL = "https://script.google.com/macros/s/AKfycbyp01BDLgzqHk5HbYt7Tl0hYESKo4qRs8AMJsFKUfbNKdbUuzjT6yb1L2qVFd_oz2Ur/exec"

def _post_event(event_type, payload=None):
    body = json.dumps({
        "event_type": event_type,
        "email":      _student_email,
        "unit":       UNIT_NUMBER,
        "payload":    payload or {}
    }).encode("utf-8")
    try:
        req = Request(TRACKER_URL, data=body,
                      headers={"Content-Type": "text/plain;charset=utf-8"})
        urlopen(req, timeout=10).read()
    except URLError as e:
        print("(could not reach tracker:", e, ")")

_student_email = input("Enter the email you signed up with: ").strip().lower()
if "@" not in _student_email:
    raise ValueError("That does not look like a valid email. Re-run this cell.")

print(f"Hi {_student_email}! Logging that you started Unit {UNIT_NUMBER}.")
_post_event("unit_started")

In [ ]:
# Standard imports for the whole chapter.
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import patches
from matplotlib.animation import FuncAnimation
from mpl_toolkits.mplot3d import Axes3D
from IPython.display import HTML, display, Markdown
import math, random

np.random.seed(11); random.seed(11)
plt.rcParams['figure.dpi'] = 100

# ---- Reusable quantum simulator (same one we built in Unit 3) ----
def ket0(n):
    s = np.zeros(2**n, dtype=complex); s[0] = 1.0
    return s

def kron_all(mats):
    out = mats[0]
    for m in mats[1:]:
        out = np.kron(out, m)
    return out

I2 = np.eye(2, dtype=complex)
X  = np.array([[0,1],[1,0]], dtype=complex)
Y  = np.array([[0,-1j],[1j,0]], dtype=complex)
Z  = np.array([[1,0],[0,-1]], dtype=complex)
H  = (1/np.sqrt(2))*np.array([[1,1],[1,-1]], dtype=complex)

def Rx(t):  c,s = np.cos(t/2), np.sin(t/2); return np.array([[c,-1j*s],[-1j*s,c]], dtype=complex)
def Ry(t):  c,s = np.cos(t/2), np.sin(t/2); return np.array([[c,-s],[s,c]],          dtype=complex)
def Rz(t):  return np.array([[np.exp(-1j*t/2),0],[0,np.exp(1j*t/2)]], dtype=complex)

def apply_1q(gate, qubit, n):
    return kron_all([gate if i==qubit else I2 for i in range(n)])

def apply_cnot(control, target, n):
    dim = 2**n
    op = np.zeros((dim, dim), dtype=complex)
    for x in range(dim):
        bits = [(x >> (n-1-i)) & 1 for i in range(n)]
        if bits[control] == 1: bits[target] ^= 1
        y = 0
        for b in bits: y = (y<<1) | b
        op[y, x] = 1
    return op

print("Quantum simulator ready.")

---
# Section 4.1 - Optimization in Machine Learning

## What is optimization, really?

Almost every machine-learning model you have ever heard of, from logistic regression to a 100-billion-parameter language model, is trained the same way: by **minimizing a loss function**. The loss measures how wrong the model is, and the optimizer's job is to find the parameter setting where that wrongness is as small as possible.

Mathematically, if your model has parameters $\theta$ and a loss $L(\theta)$, training means solving

$$ \theta^\star = \arg\min_{\theta}\, L(\theta). $$

The shape of $L(\theta)$ is called the **loss surface** (or landscape). The optimizer is essentially a hiker trying to find the lowest valley in a possibly very wrinkled terrain.

## Classical optimization methods at a glance

| Method | Idea | Where it shines |
|---|---|---|
| **Gradient Descent (GD)** | Step downhill in the steepest direction | Smooth, convex problems |
| **Stochastic Gradient Descent (SGD)** | Use a mini-batch to estimate the gradient cheaply | Large datasets, deep nets |
| **Adam** | SGD with adaptive per-parameter learning rates and momentum | Standard for deep learning |
| **Newton / quasi-Newton** | Use second derivatives (curvature) | Small-to-medium smooth problems |

All of these are *local* methods. They follow gradients (or approximations of them) downhill from wherever they start. If the loss surface has many valleys, they might get stuck in a bad one.

## Why quantum optimization?

Quantum methods give two qualitatively different angles on this problem:

1. **Quantum annealing and variational algorithms** can sometimes *tunnel* through energy barriers that block classical methods, helping escape poor local minima.
2. For certain **structured combinatorial** problems (logical, graph-based, or discrete), quantum approaches like QAOA and VQE encode the cost in a Hamiltonian and use quantum interference to make low-cost states more likely.

The catch, of course, is that quantum hardware is noisy and limited. So the **dominant near-term recipe** is the same hybrid loop we saw in Unit 3:

```
classical optimizer chooses parameters theta
        |
        v
quantum circuit evaluates cost C(theta)
        |
        v
loop until C(theta) is small
```

The quantum computer evaluates the cost, and a classical optimizer adjusts the parameters.

### Visualizing a convex vs nonconvex loss surface

A *convex* surface is bowl-shaped: any optimizer that walks downhill ends up at the global minimum. A *nonconvex* surface has multiple valleys, and the answer depends on where you start.

In [ ]:
# Two toy 2D loss surfaces side by side.
x = np.linspace(-3, 3, 100); y = np.linspace(-3, 3, 100)
X1, Y1 = np.meshgrid(x, y)

convex    = X1**2 + Y1**2                                                      # one bowl
nonconvex = (np.sin(2*X1) + np.cos(2*Y1) + 0.4*(X1**2 + Y1**2))                 # bumpy

fig = plt.figure(figsize=(11, 4))
ax1 = fig.add_subplot(1, 2, 1, projection='3d')
ax1.plot_surface(X1, Y1, convex, cmap='viridis', alpha=0.85)
ax1.set_title("Convex loss (one global minimum)")
ax1.set_xlabel(r"$\theta_1$"); ax1.set_ylabel(r"$\theta_2$"); ax1.set_zlabel("L")

ax2 = fig.add_subplot(1, 2, 2, projection='3d')
ax2.plot_surface(X1, Y1, nonconvex, cmap='magma', alpha=0.85)
ax2.set_title("Nonconvex loss (many local minima)")
ax2.set_xlabel(r"$\theta_1$"); ax2.set_ylabel(r"$\theta_2$"); ax2.set_zlabel("L")

plt.tight_layout(); plt.show()
print("Notice the bumps in the right plot. Gradient descent can get stuck in any of those dips.")

### Activity 4.1 - Optimizer trajectories on a noisy loss

Let's plot the path a simple gradient-descent run takes on a noisy 2D loss. Each step is in the negative-gradient direction, plus a bit of measurement noise (the kind a quantum circuit would have). Watch how the trajectory wanders, and how noise can both hurt (jitter) and help (escape local minima).

In [ ]:
# Noisy-gradient descent on a nonconvex 2D loss.
def loss_2d(t):
    x, y = t
    return np.sin(2*x) + np.cos(2*y) + 0.4*(x*x + y*y)

def grad_2d(t, eps=1e-3):
    # finite-difference gradient
    g = np.zeros(2)
    for i in range(2):
        tp = t.copy(); tp[i] += eps
        tm = t.copy(); tm[i] -= eps
        g[i] = (loss_2d(tp) - loss_2d(tm)) / (2*eps)
    return g

def run_optim(start, steps=80, lr=0.12, noise=0.0):
    t = np.array(start, dtype=float)
    traj = [t.copy()]
    for _ in range(steps):
        g = grad_2d(t) + noise*np.random.randn(2)
        t = t - lr * g
        traj.append(t.copy())
    return np.array(traj)

starts  = [(-2.3, 2.0), (1.6, -1.8), (-0.4, 0.5)]
noises  = [0.0, 0.3, 0.8]

x = np.linspace(-3, 3, 100); y = np.linspace(-3, 3, 100)
X1, Y1 = np.meshgrid(x, y); Zland = np.sin(2*X1) + np.cos(2*Y1) + 0.4*(X1**2 + Y1**2)

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, nz in zip(axes, noises):
    cs = ax.contourf(X1, Y1, Zland, levels=18, cmap='magma')
    for st in starts:
        traj = run_optim(st, noise=nz)
        ax.plot(traj[:,0], traj[:,1], lw=1.8, marker='o', markersize=2.5)
        ax.plot(*st, marker='s', color='white', markersize=8)
        ax.plot(*traj[-1], marker='*', color='cyan', markersize=12)
    ax.set_title(f"noise level = {nz}")
    ax.set_xlabel(r"$\theta_1$"); ax.set_ylabel(r"$\theta_2$")
plt.suptitle("Noisy gradient descent. Squares = start, stars = end.", y=1.02)
plt.tight_layout(); plt.show()
print("Higher noise jitters the path. Sometimes it helps escape a shallow valley, sometimes it hurts.")

<details>
<summary>Reflection: why does this matter for quantum?</summary>

Quantum hardware is intrinsically noisy. Every cost evaluation on a real quantum device is, in effect, a *noisy* function evaluation. So the noisy gradient descent you just visualized is closer to what hybrid quantum-classical training actually looks like than a textbook idealized run.
</details>

---
# Section 4.2 - Variational Quantum Eigensolver (VQE)

## The big idea

VQE is a hybrid algorithm for finding the **lowest eigenvalue** of a matrix called a **Hamiltonian** $H$. Why do we care? Because lots of important problems can be written as "find the input that minimizes this cost", and that cost can be packaged as a Hamiltonian.

The variational principle from quantum mechanics says: for *any* state $|\psi\rangle$,

$$ \langle\psi|H|\psi\rangle \;\geq\; E_{\min}, $$

where $E_{\min}$ is the lowest eigenvalue (the **ground-state energy**). Equality holds only when $|\psi\rangle$ is the ground state itself.

So the VQE recipe is:

1. **Parameterize** a trial state $|\psi(\theta)\rangle = U(\theta)|0\dots 0\rangle$ using a quantum circuit with knobs $\theta$.
2. **Measure** the expectation value $E(\theta) = \langle\psi(\theta)|H|\psi(\theta)\rangle$.
3. **Let a classical optimizer** push $\theta$ around to minimize $E(\theta)$.
4. At the end, $E(\theta^\star)$ is the best estimate of $E_{\min}$, and $|\psi(\theta^\star)\rangle$ is your best guess at the ground state.

## Why is this an AI tool?

In ML, the lowest-energy state of a cleverly chosen Hamiltonian can encode:
- the best assignment of training examples to clusters,
- the lowest-loss configuration of a discrete model (like a Boltzmann machine),
- the best feature subset, or
- the most-likely state of a graphical model.

So VQE is, in spirit, "loss minimization done with quantum amplitudes".

### Visualizing the energy landscape we are searching

Below is a 2-parameter VQE on a tiny 2-qubit Hamiltonian. We sweep $\theta_1, \theta_2$ over a grid and plot $E(\theta_1, \theta_2)$. The optimizer's job is to find the darkest dip on this surface.

In [ ]:
# Tiny Hamiltonian: H = Z0 Z1 + 0.5 X0 + 0.3 X1 (a toy Ising-like Hamiltonian on 2 qubits)
ZZ = kron_all([Z, Z])
X0 = kron_all([X, I2])
X1 = kron_all([I2, X])
H_op = ZZ + 0.5*X0 + 0.3*X1   # the matrix whose lowest eigenvalue we want

# True ground energy for sanity
true_E0 = np.linalg.eigvalsh(H_op)[0].real
print(f"True ground energy of H: {true_E0:+.4f}")

def trial_state(theta):
    """Two-parameter ansatz: |psi(theta1, theta2)> = (Ry(t1) x Ry(t2)) CNOT |00>."""
    t1, t2 = theta
    psi = ket0(2)
    psi = apply_1q(Ry(t1), 0, 2) @ psi
    psi = apply_1q(Ry(t2), 1, 2) @ psi
    psi = apply_cnot(0, 1, 2) @ psi
    return psi

def energy(theta):
    psi = trial_state(theta)
    return np.real(psi.conj() @ (H_op @ psi))

# Sweep the landscape
n_grid = 40
t1s = np.linspace(-np.pi, np.pi, n_grid); t2s = np.linspace(-np.pi, np.pi, n_grid)
T1, T2 = np.meshgrid(t1s, t2s)
E = np.zeros_like(T1)
for i in range(n_grid):
    for j in range(n_grid):
        E[i, j] = energy([T1[i,j], T2[i,j]])

fig, ax = plt.subplots(figsize=(6, 5))
cs = ax.contourf(T1, T2, E, levels=24, cmap='viridis')
plt.colorbar(cs, ax=ax, label=r"$E(\theta_1, \theta_2)$")
i_min, j_min = np.unravel_index(np.argmin(E), E.shape)
ax.plot(T1[i_min, j_min], T2[i_min, j_min], marker='*', color='white', markersize=14, label='argmin on grid')
ax.set_xlabel(r"$\theta_1$"); ax.set_ylabel(r"$\theta_2$")
ax.set_title("VQE energy landscape for a tiny 2-qubit Hamiltonian"); ax.legend()
plt.tight_layout(); plt.show()

### Activity 4.2 - Build a mini VQE

Let's actually run the classical optimization loop. We use simple gradient descent (with finite-difference gradients) on the same toy Hamiltonian above and watch the energy come down toward the true ground-state value.

In [ ]:
# Mini VQE: gradient descent on theta to minimize energy
def num_grad(f, theta, eps=1e-3):
    g = np.zeros_like(theta)
    for i in range(len(theta)):
        tp = theta.copy(); tp[i] += eps
        tm = theta.copy(); tm[i] -= eps
        g[i] = (f(tp) - f(tm)) / (2*eps)
    return g

theta = np.array([0.7, -1.1])     # random starting point
lr = 0.18
history = {'theta': [theta.copy()], 'E': [energy(theta)]}

for step in range(50):
    g = num_grad(energy, theta)
    theta = theta - lr * g
    history['theta'].append(theta.copy())
    history['E'].append(energy(theta))

print(f"Starting energy: {history['E'][0]:+.4f}")
print(f"Final energy:    {history['E'][-1]:+.4f}")
print(f"True minimum:    {true_E0:+.4f}")
print(f"Error from true: {history['E'][-1] - true_E0:+.4e}")

# Plot the trajectory on the landscape + energy vs step
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
cs = axes[0].contourf(T1, T2, E, levels=24, cmap='viridis')
plt.colorbar(cs, ax=axes[0])
traj = np.array(history['theta'])
axes[0].plot(traj[:,0], traj[:,1], color='white', lw=1.5)
axes[0].plot(*traj[0], marker='s', color='red',   markersize=9, label='start')
axes[0].plot(*traj[-1], marker='*', color='cyan', markersize=14, label='end')
axes[0].set_xlabel(r"$\theta_1$"); axes[0].set_ylabel(r"$\theta_2$")
axes[0].set_title("VQE descent on the landscape"); axes[0].legend()

axes[1].plot(history['E'], color='#5B2C91', lw=2, label=r'$E(\theta_t)$')
axes[1].axhline(true_E0, color='gray', ls='--', label=f'true min = {true_E0:.3f}')
axes[1].set_xlabel("step"); axes[1].set_ylabel("energy"); axes[1].legend()
axes[1].set_title("Energy comes down toward the true ground state")
plt.tight_layout(); plt.show()

<details>
<summary>Try this on your own</summary>

Change the Hamiltonian above. For example, set `H_op = ZZ - 1.5*X0 + 0.7*X1` and rerun. The landscape changes, but the same VQE recipe still finds its lowest point. That generality is exactly what makes VQE attractive: same template, many problems.
</details>

---
# Section 4.3 - Quantum Approximate Optimization Algorithm (QAOA)

## What problem does QAOA solve?

QAOA is built for **combinatorial optimization**: problems where the variables are discrete (0/1, in/out, true/false) and the goal is to find an assignment minimizing some cost.

Examples close to AI:
- **Feature selection** - pick a subset of features that maximizes information about a target.
- **Graph partitioning / clustering** - split a graph into balanced groups while cutting as few edges as possible.
- **MAP inference** in graphical models - find the most likely joint assignment.
- **Max-Cut** - the textbook example, and the one we will demo.

## The core idea

Like VQE, QAOA is a variational, hybrid algorithm. The differences:

1. The cost is encoded in a **cost Hamiltonian** $H_C$ whose ground state is the optimal bit-string.
2. The ansatz alternates between two unitaries: $e^{-i\gamma H_C}$ (cost) and $e^{-i\beta H_M}$ (mixer), repeated $p$ times.
3. The parameters are $\vec\gamma, \vec\beta$, which are tuned classically.

In one slogan: **QAOA "stirs" amplitude toward low-cost bit-strings**, repeatedly, with a classical optimizer choosing how much stirring to do at each layer.

### Demo: Max-Cut on a tiny graph

We will solve Max-Cut on a 4-node graph by brute force first (to know the answer), then walk through how QAOA would sample it.

In [ ]:
# A small 4-node graph
edges = [(0,1), (1,2), (2,3), (3,0), (0,2)]    # 4-cycle plus a diagonal
n_nodes = 4
N = 2**n_nodes

def cut_value(bits, edges):
    # bits is a length-n tuple of 0/1; cut counts edges between groups
    return sum(1 for (u,v) in edges if bits[u] != bits[v])

# Brute-force all 16 assignments
all_bits = [tuple((i >> (n_nodes-1-k)) & 1 for k in range(n_nodes)) for i in range(N)]
cuts = [cut_value(b, edges) for b in all_bits]
best_cut = max(cuts)
best_assignments = [b for b, c in zip(all_bits, cuts) if c == best_cut]

print(f"Best cut value: {best_cut}")
print("Optimal assignments:")
for b in best_assignments:
    print(" ", b)

# Draw the graph and the best cut
import math
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
def node_pos(n_nodes):
    return {i: (math.cos(2*math.pi*i/n_nodes), math.sin(2*math.pi*i/n_nodes)) for i in range(n_nodes)}
pos = node_pos(n_nodes)

# Left: the graph
for (u,v) in edges:
    axes[0].plot([pos[u][0], pos[v][0]], [pos[u][1], pos[v][1]], color='gray', lw=2)
for i,(x,y) in pos.items():
    axes[0].add_patch(patches.Circle((x,y), 0.12, color='#5B2C91'))
    axes[0].text(x, y, str(i), color='white', ha='center', va='center', fontweight='bold')
axes[0].set_xlim(-1.3,1.3); axes[0].set_ylim(-1.3,1.3); axes[0].set_aspect('equal')
axes[0].set_title("The graph"); axes[0].axis('off')

# Right: one optimal cut, color nodes by group
chosen = best_assignments[0]
group_colors = ['#2A9D8F', '#E76F51']
for (u,v) in edges:
    color = 'red' if chosen[u] != chosen[v] else 'gray'
    axes[1].plot([pos[u][0], pos[v][0]], [pos[u][1], pos[v][1]], color=color, lw=2)
for i,(x,y) in pos.items():
    axes[1].add_patch(patches.Circle((x,y), 0.12, color=group_colors[chosen[i]]))
    axes[1].text(x, y, str(i), color='white', ha='center', va='center', fontweight='bold')
axes[1].set_xlim(-1.3,1.3); axes[1].set_ylim(-1.3,1.3); axes[1].set_aspect('equal')
axes[1].set_title(f"Optimal Max-Cut, value = {best_cut}\nGreen vs orange = the two sides")
axes[1].axis('off')
plt.tight_layout(); plt.show()

### A minimal QAOA simulation

We will simulate a one-layer ($p=1$) QAOA on this graph by directly computing the cost-amplified state. The point is to *see the distribution sharpen* toward good cuts as we tune the angles.

In [ ]:
# Encode each edge as a Z_u Z_v term in the cost Hamiltonian.
# H_C |bits> = sum_{(u,v) in E} (1 - z_u z_v)/2 |bits>, where z_i = +/-1 from bit i.
def cost_diag(edges, n_nodes):
    diag = np.zeros(2**n_nodes)
    for idx in range(2**n_nodes):
        bits = [(idx >> (n_nodes-1-k)) & 1 for k in range(n_nodes)]
        diag[idx] = sum(1 for (u,v) in edges if bits[u] != bits[v])
    return diag

C_diag = cost_diag(edges, n_nodes)
max_C  = C_diag.max()                              # = best_cut

# One layer of QAOA on uniform superposition:
# 1) start in equal superposition |s>
# 2) apply U_C(gamma) = diag(exp(-i*gamma*C))
# 3) apply mixer U_M(beta) = (Rx(2*beta))^{otimes n}
def qaoa_state(gamma, beta, n_nodes, C_diag):
    s = np.ones(2**n_nodes, dtype=complex) / np.sqrt(2**n_nodes)
    s = s * np.exp(-1j * gamma * C_diag)
    mix = kron_all([Rx(2*beta)]*n_nodes)
    return mix @ s

def expected_cost(gamma, beta, n_nodes, C_diag):
    psi = qaoa_state(gamma, beta, n_nodes, C_diag)
    return np.real(np.sum(C_diag * np.abs(psi)**2))

# Sweep (gamma, beta) to find good values
gs = np.linspace(0, np.pi, 40); bs = np.linspace(0, np.pi/2, 40)
G, B = np.meshgrid(gs, bs)
EC = np.zeros_like(G)
for i in range(len(bs)):
    for j in range(len(gs)):
        EC[i, j] = expected_cost(G[i,j], B[i,j], n_nodes, C_diag)

best_idx = np.unravel_index(np.argmax(EC), EC.shape)
g_star, b_star = G[best_idx], B[best_idx]
psi_star = qaoa_state(g_star, b_star, n_nodes, C_diag)
probs    = np.abs(psi_star)**2

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
cs = axes[0].contourf(G, B, EC, levels=20, cmap='viridis')
plt.colorbar(cs, ax=axes[0], label='expected cut value')
axes[0].plot(g_star, b_star, marker='*', color='white', markersize=14)
axes[0].set_xlabel(r"$\gamma$"); axes[0].set_ylabel(r"$\beta$")
axes[0].set_title(f"QAOA landscape, best (gamma,beta)=({g_star:.2f}, {b_star:.2f})")

bars = axes[1].bar(range(N), probs, color=['#2A9D8F' if C_diag[k] == max_C else '#bbbbbb' for k in range(N)])
axes[1].set_xticks(range(N))
axes[1].set_xticklabels([''.join(str(b) for b in all_bits[i]) for i in range(N)], rotation=70, fontsize=8)
axes[1].set_ylabel("probability"); axes[1].set_title("Measurement distribution at best QAOA angles\nGreen bars = optimal cuts")
plt.tight_layout(); plt.show()

# Probability the QAOA sample is optimal
p_opt = sum(probs[i] for i in range(N) if C_diag[i] == max_C)
print(f"Probability that one QAOA measurement returns an OPTIMAL cut: {p_opt:.3f}")
print(f"Random guessing baseline:                                  {len(best_assignments)/N:.3f}")

### Activity 4.3 - Edit the graph and re-run

QAOA is template-style: change the graph, the same code finds the new best cut.

Try modifying the `edges` list below to make a different graph (or a different problem entirely - any cost function works as long as you can write a diagonal `C_diag`).

In [ ]:
# ---- Activity 4.3: experiment with another graph ----
edges_new = [(0,1), (1,2), (2,3), (3,0)]   # plain 4-cycle, no diagonal
C_diag_new = cost_diag(edges_new, n_nodes)
max_C_new  = C_diag_new.max()
all_bits_new = [tuple((i >> (n_nodes-1-k)) & 1 for k in range(n_nodes)) for i in range(N)]
best_new = [b for b,c in zip(all_bits_new, C_diag_new) if c == max_C_new]
print(f"New best cut value: {max_C_new}")
print("Optimal assignments:", best_new)

# Sweep and pick best (gamma, beta)
EC2 = np.zeros_like(G)
for i in range(len(bs)):
    for j in range(len(gs)):
        EC2[i, j] = expected_cost(G[i,j], B[i,j], n_nodes, C_diag_new)
bi = np.unravel_index(np.argmax(EC2), EC2.shape)
psi2 = qaoa_state(G[bi], B[bi], n_nodes, C_diag_new)
probs2 = np.abs(psi2)**2

fig, ax = plt.subplots(figsize=(10, 3.4))
colors = ['#2A9D8F' if C_diag_new[k] == max_C_new else '#bbbbbb' for k in range(N)]
ax.bar(range(N), probs2, color=colors)
ax.set_xticks(range(N))
ax.set_xticklabels([''.join(str(b) for b in all_bits_new[i]) for i in range(N)], rotation=70, fontsize=8)
ax.set_ylabel("probability"); ax.set_title(f"QAOA on the new graph, gamma={G[bi]:.2f}, beta={B[bi]:.2f}")
plt.tight_layout(); plt.show()

---
# Section 4.4 - Applications to AI Systems

Where does all of this actually plug into an AI pipeline? Here is a tour, with the underlying optimization in each case.

| AI use case | What is being optimized | Quantum tool |
|---|---|---|
| **Feature selection** | Choose subset of features minimizing prediction error / max info | QAOA on a QUBO over features |
| **Clustering / community detection** | Partition a similarity graph to minimize cut + balance constraints | QAOA / Max-Cut |
| **Boltzmann machine training** | Sample low-energy configurations | VQE + quantum sampling |
| **Hyperparameter search** | Find best discrete combination of training settings | QAOA, Grover-style search |
| **Portfolio / resource allocation** | Pick assets maximizing return for given risk | QAOA on a QUBO |
| **Causal / Bayesian-network inference (MAP)** | Find most likely joint assignment | VQE on a tailored Hamiltonian |
| **Constraint satisfaction in planning** | Find variable assignment satisfying constraints | QAOA, Grover |
| **Loss-landscape escape in deep learning (research)** | Use quantum tunneling to find better minima | Quantum annealing |

## A practical pattern

Most AI applications follow this template:

1. **Encode** the objective as a cost function over discrete variables (a Quadratic Unconstrained Binary Optimization, or QUBO).
2. Translate the QUBO into a **cost Hamiltonian** $H_C$.
3. Pick a variational algorithm: usually **QAOA** for combinatorial problems, **VQE** for continuous or chemistry-flavored ones.
4. Run the hybrid loop: quantum circuit evaluates cost, classical optimizer adjusts parameters.
5. Sample the final state and pick the best bit-string seen.

## Activity 4.4 - Feature selection as Max-Cut-like QUBO

Imagine you have 4 candidate features for a model. Some pairs of features are **redundant** with each other, so including both wastes a feature slot. We will:

1. Build a small "redundancy" graph between features.
2. Compute the QUBO whose ground state selects a *diverse* set of features.
3. Solve it via the same QAOA code we already have.

In [ ]:
# ---- Activity 4.4: feature selection ----
# Score for each individual feature (how informative it is on its own, made up here).
feature_score = np.array([0.9, 0.7, 0.6, 0.8])      # higher = more useful

# Redundancy between feature pairs (1 means highly redundant, 0 means independent).
# rows / cols correspond to features 0..3
R = np.array([
    [0.0, 0.7, 0.1, 0.0],
    [0.7, 0.0, 0.0, 0.2],
    [0.1, 0.0, 0.0, 0.8],
    [0.0, 0.2, 0.8, 0.0],
])

# Define reward(s) = (sum of scores for selected) - lambda * (sum of redundancies between selected)
def feature_reward(bits):
    sel = [i for i,b in enumerate(bits) if b == 1]
    score = sum(feature_score[i] for i in sel)
    redun = sum(R[i,j] for i in sel for j in sel if i < j)
    return score - 1.2 * redun

all_bits_f = [tuple((i >> (n_nodes-1-k)) & 1 for k in range(n_nodes)) for i in range(N)]
rewards = np.array([feature_reward(b) for b in all_bits_f])
best_r = rewards.max()
best_sets = [b for b,r in zip(all_bits_f, rewards) if r == best_r]

print(f"Best feature-selection reward: {best_r:.3f}")
print("Optimal feature sets (1 = include feature):")
for b in best_sets:
    sel = [i for i,bit in enumerate(b) if bit == 1]
    print(f"  bits={b}  features={sel}")

# Plug into QAOA: we want to MAXIMIZE reward, so cost diag is the reward itself.
C_diag_feat = rewards.copy()

# Sweep angles
EC3 = np.zeros_like(G)
for i in range(len(bs)):
    for j in range(len(gs)):
        EC3[i, j] = expected_cost(G[i,j], B[i,j], n_nodes, C_diag_feat)
bi = np.unravel_index(np.argmax(EC3), EC3.shape)
psi3 = qaoa_state(G[bi], B[bi], n_nodes, C_diag_feat)
probs3 = np.abs(psi3)**2

fig, ax = plt.subplots(figsize=(10, 3.4))
colors = ['#2A9D8F' if rewards[k] == best_r else '#bbbbbb' for k in range(N)]
ax.bar(range(N), probs3, color=colors)
ax.set_xticks(range(N))
ax.set_xticklabels([''.join(str(b) for b in all_bits_f[i]) for i in range(N)], rotation=70, fontsize=8)
ax.set_ylabel("probability"); ax.set_title(f"QAOA on feature-selection QUBO, gamma={G[bi]:.2f}, beta={B[bi]:.2f}")
plt.tight_layout(); plt.show()
print("Green bar = an optimal feature set. QAOA concentrates measurement probability there.")

---
# End-of-Unit Quiz

Try each question first. Click the arrow to reveal the answer.

## Section A: Multiple Choice

**1. In machine learning, training a model is essentially:**

A) Sampling from a uniform distribution.
B) Minimizing a loss function over model parameters.
C) Maximizing the entropy of the dataset.
D) Computing the inverse of the data matrix.

<details><summary>Answer 1</summary>**B).** All training methods, from SGD to Adam, are minimizing some loss $L(\theta)$.</details>

---

**2. The "hybrid quantum-classical loop" used in VQE and QAOA means:**

A) The quantum computer runs the optimizer; the classical computer just stores qubits.
B) Each step alternates a quantum cost evaluation with a classical parameter update.
C) Some qubits are classical and some are quantum.
D) The algorithm runs half on a CPU and half on a GPU.

<details><summary>Answer 2</summary>**B).** The quantum device evaluates the cost; a classical optimizer adjusts the variational parameters.</details>

---

**3. The variational principle says that for any trial state $|\psi\rangle$:**

A) $\langle\psi|H|\psi\rangle = E_{\min}$ always.
B) $\langle\psi|H|\psi\rangle \geq E_{\min}$, with equality only at the true ground state.
C) $\langle\psi|H|\psi\rangle$ is independent of $H$.
D) $\langle\psi|H|\psi\rangle$ is negative.

<details><summary>Answer 3</summary>**B).** This is why VQE works: minimizing the expectation value approaches the true ground energy from above.</details>

---

**4. Which problem is QAOA *best suited* for?**

A) Inverting a dense matrix.
B) Estimating a continuous integral.
C) Combinatorial optimization (discrete variables, e.g. Max-Cut, feature selection).
D) Training a deep convolutional network end-to-end.

<details><summary>Answer 4</summary>**C).** QAOA shines on discrete optimization problems encoded as cost Hamiltonians.</details>

---

**5. What does the "mixer" $e^{-i\beta H_M}$ in QAOA do?**

A) It does nothing; it is just notation.
B) It rotates each qubit so that amplitude can flow between basis states; this is how good bit-strings can become more probable.
C) It measures all qubits.
D) It prepares the initial state.

<details><summary>Answer 5</summary>**B).** The mixer is what lets QAOA *redistribute* amplitude. Without it, the cost unitary would only apply phases.</details>

## Section B: Short Answer

**6. Give two reasons noise in a quantum optimizer is not always bad.**

<details><summary>Answer 6</summary>Two valid reasons: (a) noise jitters the trajectory and can help it escape shallow local minima, similar to SGD's mini-batch noise in deep learning; (b) some forms of noise can implicitly regularize the variational parameters, helping generalization.</details>

---

**7. Explain in 2 sentences how VQE turns "find the ground state of $H$" into a classical optimization problem.**

<details><summary>Answer 7</summary>VQE parameterizes a trial state $|\psi(\theta)\rangle$ by a quantum circuit with knobs $\theta$. The expectation $E(\theta) = \langle\psi(\theta)|H|\psi(\theta)\rangle$ is a classical function of $\theta$, so a classical optimizer (Adam, COBYLA, gradient descent) can minimize it, and by the variational principle the minimum approaches the true ground-state energy.</details>

---

**8. Why is feature selection naturally a discrete optimization problem, and how does that connect to QAOA?**

<details><summary>Answer 8</summary>Each feature is either included or excluded, so the variable for each feature is a bit (0 or 1). The total reward is a function of the chosen subset, exactly the kind of discrete cost QAOA was designed for: encode the reward as a cost Hamiltonian whose diagonal entries are the reward values, then let QAOA concentrate measurement probability on the high-reward bit-strings.</details>

---

**9. Name one piece of an actual deep-learning workflow where a quantum optimizer could plausibly help, and say why.**

<details><summary>Answer 9</summary>Any of: discrete hyperparameter search (which optimizer, batch size, augmentations to use); pruning a neural network (which weights to keep); architecture search over a discrete set of layer choices; Boltzmann machine training (sampling low-energy configurations); or finding a good initialization seed. In all of these the underlying problem is discrete or combinatorial, which is where QAOA/quantum annealing have potential leverage.</details>

---

**10. (Bonus) In your own words, why are VQE and QAOA "two sides of the same coin"?**

<details><summary>Answer 10</summary>Both are variational hybrid algorithms: prepare a parameterized quantum state, measure an expectation value with respect to a problem Hamiltonian, and use a classical optimizer to tune the parameters. VQE typically targets the *lowest eigenvalue* of a Hamiltonian (continuous, chemistry-like), and QAOA targets the *lowest cost bit-string* of a combinatorial problem encoded as a Hamiltonian. The math (variational principle, hybrid loop) is the same; the choice of ansatz and Hamiltonian is what differentiates them.</details>

---

## You're done!

You now have a working mental model of how quantum optimization is used in AI:

1. **Loss minimization** is at the heart of all of ML.
2. **VQE** finds minimum-energy states of Hamiltonians, useful when your AI cost can be packaged that way.
3. **QAOA** is the go-to for combinatorial optimization in AI: feature selection, graph problems, MAP inference.
4. **Real applications** plug into many parts of an ML pipeline, from preprocessing (feature selection) to training (sampling) to architecture search.

---
## End-of-unit submission

Fill in your quiz answers below, then run this cell to submit. Your answers and the fact that you finished Unit 4 are logged automatically. After your 5th completed unit, your certificate will be emailed to you.

In [ ]:
# ============================================================
# END-OF-UNIT SUBMISSION - fill in your answers below
# ============================================================

quiz_answers = {
    "q1": "",        # multiple choice: A, B, C, or D
    "q2": "",
    "q3": "",
    "q4": "",
    "q5": "",
    "q6_short_answer": "Type your answer to question 6 here.",
    "q7_short_answer": "Type your answer to question 7 here.",
    "q8_short_answer": "Type your answer to question 8 here.",
    "q9_short_answer": "Type your answer to question 9 here.",
    "q10_bonus":       "Type your bonus answer here, or leave blank."
}

reflection = "What did you find most interesting in this unit? (optional)"

_post_event("unit_completed",
            payload={"quiz": quiz_answers, "reflection": reflection})

print(f"Submitted Unit 4!")
print("If this was your 5th completed unit, watch your inbox for a certificate.")